# 02 — Controle fixo para DiD (com holdout temporal)
Princípios de design:
1. **Holdout temporal**: otimização roda nos primeiros 70% da janela; tendências
   paralelas são testadas **apenas nos 30% finais** (critério de aceite, nunca
   de otimização) — elimina pre-testing contamination.
2. **p-value fora do score**: score = correlação + forma + região + ruptura.
3. **Target limpo**: Brasil excluindo o próprio controle e tratadas ativas.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-2.0.0a2-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.spark_io import (load_city_panel, TABLES,
                                          active_blocked_cities, NATIONAL_HOLIDAYS)
from supply_experiments.design.control_selection import select_fixed_control

TARGET_ENV = "sandbox"
tables = TABLES[TARGET_ENV]
# Só ative após confirmar que ausência de linha significa zero atividade.
ASSUME_MISSING_CITY_DAYS_ZERO = False
panel, stats = load_city_panel(
    spark, "2025-09-01", "2026-06-30",
    assume_missing_city_days_zero=ASSUME_MISSING_CITY_DAYS_ZERO,
    require_rupture_telemetry=True,  # ruptura participa do score de seleção
)

blocked = active_blocked_cities(spark, tables["registry"])
print("Bloqueadas (experimentos ativos):", blocked)

In [ ]:
STATE_TO_REGION = {
    "SP":"Sudeste","RJ":"Sudeste","MG":"Sudeste","ES":"Sudeste",
    "PR":"Sul","SC":"Sul","RS":"Sul",
    "BA":"Nordeste","PE":"Nordeste","CE":"Nordeste","MA":"Nordeste","PB":"Nordeste",
    "RN":"Nordeste","AL":"Nordeste","SE":"Nordeste","PI":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
    "AM":"Norte","PA":"Norte","RO":"Norte","RR":"Norte","AC":"Norte","AP":"Norte","TO":"Norte",
}
candidates = [c for c in stats.index
              if c in panel.cities
              and c not in blocked["treated_active"]]
city_states  = stats["state_address"].to_dict()
city_rupture = stats["rupture_rate"].to_dict()

res = select_fixed_control(
    panel, candidates, city_states, STATE_TO_REGION,
    max_cities=9, train_frac=0.7, alpha=0.05,
    holidays=NATIONAL_HOLIDAYS,
    city_rupture=city_rupture,
    target_exclude=set(blocked["treated_active"]),
)
print("Cidades:", res.cities)
print(f"Score treino: {res.train_score:.3f}")
print(f"HOLDOUT — p-value de equivalência: {res.holdout_p_value:.3f} "
      f"({'ACEITO' if res.holdout_passed else 'REPROVADO'}) | corr: {res.holdout_corr:.3f}")

In [ ]:
# persiste no registry v2 apenas se aprovado no holdout
from supply_experiments.spark_io import ExperimentRecord, ensure_registry, save_experiment
import uuid
from datetime import date

assert res.holdout_passed, "Controle reprovado no holdout — refaça a seleção (mais candidatas / outra janela)"

rec = ExperimentRecord(
    experiment_id=str(uuid.uuid4()), name="fixed_control_groceries",
    hypothesis="Conjunto de controle fixo representativo do Brasil (uso geral DiD)",
    status="approved", design_method="did", primary_kpi="gmv",
    guardrail_kpis=["rupture_rate_order"],
    start_date=date.today(), end_date=date.today(), pre_window_days=0,
    treated_units=[],
    control_units=[{"city_norm": c, "city_address": c,
                    "state_address": city_states.get(c, ""), "weight": None}
                   for c in res.cities],
    mde_estimated=0.0, expected_effect=None,
    decision_rule="n/a (controle fixo, não é experimento)",
    created_by=spark.sql("SELECT current_user()").first()[0],
)
ensure_registry(spark, tables["registry"])
save_experiment(spark, tables["registry"], rec, require_approval_checks=False)
print("Controle fixo registrado:", rec.experiment_id)